# Holiday Package Prediction using. Multiple Models and implementating Random Forest

## Holiday Package Prediciton

### 1) Problem statement.
"Trips & Travel.Com" company wants to enable and establish a viable business model to expand the customer base.
One of the ways to expand the customer base is to introduce a new offering of packages. Currently, there are 5 types of packages the company is offering * Basic, Standard, Deluxe, Super Deluxe, King. Looking at the data of the last year, we observed that 18% of the customers purchased the packages. However, the marketing cost was quite high because customers were contacted at random without looking at the available information.
The company is now planning to launch a new product i.e. Wellness Tourism Package. Wellness Tourism is defined as Travel that allows the traveler to maintain, enhance or kick-start a healthy lifestyle, and support or increase one's sense of well-being.
However, this time company wants to harness the available data of existing and potential customers to make the marketing expenditure more efficient.
### 2) Data Collection.
The Dataset is collected from https://www.kaggle.com/datasets/susant4learning/holiday-package-purchase-prediction
The data consists of 20 column and 4888 rows.


In [2]:
# import all the necessary librariess
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import plotly.express as px

from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold,cross_val_predict

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix

In [3]:
# Load the dataset
dataset=pd.read_csv('dataset/Travel.csv')

In [4]:
dataset.sample(5)

,CustomerID,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
802,200802,0,33.0,Company Invited,3,NaN,Salaried,Female,1,3.0,Deluxe,4.0,Divorced,1.0,0,4,1,0.0,Manager,NaN
2230,202230,0,53.0,Company Invited,3,8.0,Small Business,Female,2,4.0,Standard,4.0,Married,3.0,0,1,1,0.0,Senior Manager,22525.0
3585,203585,0,31.0,Self Enquiry,1,14.0,Small Business,Male,3,5.0,Basic,4.0,Married,3.0,0,5,0,1.0,Executive,20819.0
3307,203307,0,34.0,Self Enquiry,2,9.0,Salaried,Male,3,4.0,Basic,3.0,Divorced,3.0,0,2,0,1.0,Executive,22278.0
34,200034,1,24.0,Self Enquiry,1,6.0,Small Business,Male,3,3.0,Basic,3.0,Divorced,3.0,1,3,1,2.0,Executive,17293.0


In [5]:
# Data Cleaning
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4888 entries, 0 to 4887
Data columns (total 20 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   CustomerID                4888 non-null   int64  
 1   ProdTaken                 4888 non-null   int64  
 2   Age                       4662 non-null   float64
 3   TypeofContact             4863 non-null   object 
 4   CityTier                  4888 non-null   int64  
 5   DurationOfPitch           4637 non-null   float64
 6   Occupation                4888 non-null   object 
 7   Gender                    4888 non-null   object 
 8   NumberOfPersonVisiting    4888 non-null   int64  
 9   NumberOfFollowups         4843 non-null   float64
 10  ProductPitched            4888 non-null   object 
 11  PreferredPropertyStar     4862 non-null   float64
 12  MaritalStatus             4888 non-null   object 
 13  NumberOfTrips             4748 non-null   float64
 14  Passport

In [6]:
dataset.isna().sum()

CustomerID                    0
ProdTaken                     0
Age                         226
TypeofContact                25
CityTier                      0
DurationOfPitch             251
Occupation                    0
Gender                        0
NumberOfPersonVisiting        0
NumberOfFollowups            45
ProductPitched                0
PreferredPropertyStar        26
MaritalStatus                 0
NumberOfTrips               140
Passport                      0
PitchSatisfactionScore        0
OwnCar                        0
NumberOfChildrenVisiting     66
Designation                   0
MonthlyIncome               233
dtype: int64

In [7]:
dataset[dataset.isna().any(axis=1)]

,CustomerID,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
4,200004,0,NaN,Self Enquiry,1,8.0,Small Business,Male,2,3.0,Basic,4.0,Divorced,1.0,0,5,1,0.0,Executive,18468.0
11,200011,0,NaN,Self Enquiry,1,21.0,Salaried,Female,2,4.0,Deluxe,3.0,Single,1.0,1,3,0,0.0,Manager,NaN
19,200019,0,NaN,Self Enquiry,1,8.0,Salaried,Male,2,3.0,Basic,3.0,Single,6.0,1,4,0,1.0,Executive,NaN
20,200020,0,NaN,Company Invited,1,17.0,Salaried,Female,3,2.0,Deluxe,3.0,Married,1.0,0,3,1,2.0,Manager,NaN
21,200021,1,NaN,Self Enquiry,3,15.0,Salaried,Male,2,4.0,Deluxe,5.0,Single,1.0,0,2,0,0.0,Manager,18407.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4850,204850,1,46.0,Self Enquiry,3,8.0,Salaried,Male,4,5.0,Deluxe,5.0,Married,NaN,0,4,1,3.0,Manager,36739.0
4851,204851,1,40.0,Self Enquiry,1,9.0,Salaried,Female,4,4.0,Basic,5.0,Married,NaN,1,1,1,1.0,Executive,35801.0
4868,204868,1,43.0,Company Invited,2,15.0,Salaried,Female,4,5.0,Basic,3.0,Married,NaN,0,5,1,2.0,Executive,36539.0
4869,204869,1,56.0,Self Enquiry,3,16.0,Small Business,Female,3,6.0,Basic,4.0,Single,NaN,0,1,1,2.0,Executive,37865.0


In [8]:
dataset.duplicated().sum()

np.int64(0)

In [9]:
# Split to categorical and numerical columns
categorical_cols=dataset.select_dtypes(include='O').columns
numerical_cols=dataset.select_dtypes(exclude='O').columns

In [10]:
dataset.sample(5)

,CustomerID,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
790,200790,0,33.0,Self Enquiry,1,11.0,Salaried,Male,2,4.0,Standard,3.0,Married,5.0,0,3,0,0.0,Senior Manager,22119.0
4625,204625,1,30.0,Self Enquiry,1,17.0,Salaried,Female,4,5.0,Basic,5.0,Single,8.0,1,5,1,1.0,Executive,21082.0
4772,204772,0,54.0,Self Enquiry,1,14.0,Small Business,Female,3,4.0,King,3.0,Married,NaN,0,1,1,2.0,VP,37284.0
2224,202224,0,33.0,Company Invited,3,14.0,Salaried,Female,2,4.0,Basic,3.0,Single,1.0,0,1,0,1.0,Executive,17342.0
424,200424,0,57.0,Self Enquiry,3,35.0,Small Business,Male,2,4.0,Super Deluxe,3.0,Married,4.0,0,3,0,0.0,AVP,29118.0


In [11]:
categorical_cols

Index(['TypeofContact', 'Occupation', 'Gender', 'ProductPitched',
       'MaritalStatus', 'Designation'],
      dtype='object')

In [12]:
for col in categorical_cols:
    print('---------------------')
    print(dataset[col].value_counts())
    print('--------------------')

---------------------
TypeofContact
Self Enquiry       3444
Company Invited    1419
Name: count, dtype: int64
--------------------
---------------------
Occupation
Salaried          2368
Small Business    2084
Large Business     434
Free Lancer          2
Name: count, dtype: int64
--------------------
---------------------
Gender
Male       2916
Female     1817
Fe Male     155
Name: count, dtype: int64
--------------------
---------------------
ProductPitched
Basic           1842
Deluxe          1732
Standard         742
Super Deluxe     342
King             230
Name: count, dtype: int64
--------------------
---------------------
MaritalStatus
Married      2340
Divorced      950
Single        916
Unmarried     682
Name: count, dtype: int64
--------------------
---------------------
Designation
Executive         1842
Manager           1732
Senior Manager     742
AVP                342
VP                 230
Name: count, dtype: int64
--------------------


In [13]:
dataset['Gender']=dataset['Gender'].str.replace('Fe Male','Female')

In [14]:
for col in categorical_cols:
    print('---------------------')
    print(dataset[col].value_counts())
    print('--------------------')

---------------------
TypeofContact
Self Enquiry       3444
Company Invited    1419
Name: count, dtype: int64
--------------------
---------------------
Occupation
Salaried          2368
Small Business    2084
Large Business     434
Free Lancer          2
Name: count, dtype: int64
--------------------
---------------------
Gender
Male      2916
Female    1972
Name: count, dtype: int64
--------------------
---------------------
ProductPitched
Basic           1842
Deluxe          1732
Standard         742
Super Deluxe     342
King             230
Name: count, dtype: int64
--------------------
---------------------
MaritalStatus
Married      2340
Divorced      950
Single        916
Unmarried     682
Name: count, dtype: int64
--------------------
---------------------
Designation
Executive         1842
Manager           1732
Senior Manager     742
AVP                342
VP                 230
Name: count, dtype: int64
--------------------


In [15]:
dataset['MaritalStatus']=dataset['MaritalStatus'].str.replace('Single','Unmarried')

In [16]:
for col in categorical_cols:
    print('---------------------')
    print(dataset[col].value_counts())
    print('--------------------')

---------------------
TypeofContact
Self Enquiry       3444
Company Invited    1419
Name: count, dtype: int64
--------------------
---------------------
Occupation
Salaried          2368
Small Business    2084
Large Business     434
Free Lancer          2
Name: count, dtype: int64
--------------------
---------------------
Gender
Male      2916
Female    1972
Name: count, dtype: int64
--------------------
---------------------
ProductPitched
Basic           1842
Deluxe          1732
Standard         742
Super Deluxe     342
King             230
Name: count, dtype: int64
--------------------
---------------------
MaritalStatus
Married      2340
Unmarried    1598
Divorced      950
Name: count, dtype: int64
--------------------
---------------------
Designation
Executive         1842
Manager           1732
Senior Manager     742
AVP                342
VP                 230
Name: count, dtype: int64
--------------------


In [17]:
dataset.sample(5)

,CustomerID,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
4709,204709,0,38.0,Self Enquiry,3,17.0,Salaried,Female,3,5.0,Deluxe,3.0,Married,4.0,1,4,1,2.0,Manager,25372.0
733,200733,0,26.0,Self Enquiry,1,8.0,Small Business,Male,2,1.0,Deluxe,3.0,Divorced,2.0,0,3,1,1.0,Manager,20472.0
2088,202088,0,NaN,Self Enquiry,1,8.0,Small Business,Male,3,1.0,Basic,5.0,Unmarried,1.0,0,1,0,1.0,Executive,18424.0
4864,204864,1,32.0,Self Enquiry,2,29.0,Salaried,Male,3,6.0,Basic,3.0,Married,3.0,0,1,0,2.0,Executive,28530.0
4335,204335,1,30.0,Self Enquiry,1,32.0,Large Business,Female,4,5.0,Basic,4.0,Married,7.0,0,3,1,3.0,Executive,21224.0


In [18]:
dataset['Age'] = dataset['Age'].fillna(dataset['Age'].median())
dataset['TypeofContact'] = dataset['TypeofContact'].fillna(dataset['TypeofContact'].mode()[0])
dataset['DurationOfPitch'] = dataset['DurationOfPitch'].fillna(dataset['DurationOfPitch'].median())
dataset['NumberOfFollowups'] = dataset['NumberOfFollowups'].fillna(dataset['NumberOfFollowups'].mode()[0])
dataset['PreferredPropertyStar'] = dataset['PreferredPropertyStar'].fillna(dataset['PreferredPropertyStar'].mode()[0])
dataset['NumberOfTrips'] = dataset['NumberOfTrips'].fillna(dataset['NumberOfTrips'].median())
dataset['NumberOfChildrenVisiting'] = dataset['NumberOfChildrenVisiting'].fillna(dataset['NumberOfChildrenVisiting'].mode()[0])
dataset['MonthlyIncome'] = dataset['MonthlyIncome'].fillna(dataset['MonthlyIncome'].median())


In [19]:
dataset.sample(5)

,CustomerID,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
1359,201359,0,34.0,Self Enquiry,3,9.0,Small Business,Female,3,4.0,Deluxe,3.0,Married,4.0,1,3,0,0.0,Manager,23103.0
780,200780,1,28.0,Company Invited,1,30.0,Large Business,Male,3,4.0,Standard,5.0,Unmarried,2.0,0,2,0,0.0,Senior Manager,23722.0
4401,204401,0,21.0,Self Enquiry,1,7.0,Salaried,Female,3,5.0,Basic,4.0,Unmarried,3.0,0,4,0,2.0,Executive,21514.0
2023,202023,0,40.0,Self Enquiry,1,8.0,Small Business,Female,2,4.0,Basic,3.0,Unmarried,1.0,1,3,0,0.0,Executive,17342.0
3875,203875,0,33.0,Self Enquiry,3,7.0,Small Business,Female,3,4.0,Standard,3.0,Married,3.0,0,5,1,1.0,Senior Manager,29100.0


In [20]:
dataset.isna().sum()

CustomerID                  0
ProdTaken                   0
Age                         0
TypeofContact               0
CityTier                    0
DurationOfPitch             0
Occupation                  0
Gender                      0
NumberOfPersonVisiting      0
NumberOfFollowups           0
ProductPitched              0
PreferredPropertyStar       0
MaritalStatus               0
NumberOfTrips               0
Passport                    0
PitchSatisfactionScore      0
OwnCar                      0
NumberOfChildrenVisiting    0
Designation                 0
MonthlyIncome               0
dtype: int64

In [21]:
categorical_cols

Index(['TypeofContact', 'Occupation', 'Gender', 'ProductPitched',
       'MaritalStatus', 'Designation'],
      dtype='object')

In [22]:
dataset

,CustomerID,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
0,200000,1,41.0,Self Enquiry,3,6.0,Salaried,Female,3,3.0,Deluxe,3.0,Unmarried,1.0,1,2,1,0.0,Manager,20993.0
1,200001,0,49.0,Company Invited,1,14.0,Salaried,Male,3,4.0,Deluxe,4.0,Divorced,2.0,0,3,1,2.0,Manager,20130.0
2,200002,1,37.0,Self Enquiry,1,8.0,Free Lancer,Male,3,4.0,Basic,3.0,Unmarried,7.0,1,3,0,0.0,Executive,17090.0
3,200003,0,33.0,Company Invited,1,9.0,Salaried,Female,2,3.0,Basic,3.0,Divorced,2.0,1,5,1,1.0,Executive,17909.0
4,200004,0,36.0,Self Enquiry,1,8.0,Small Business,Male,2,3.0,Basic,4.0,Divorced,1.0,0,5,1,0.0,Executive,18468.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4883,204883,1,49.0,Self Enquiry,3,9.0,Small Business,Male,3,5.0,Deluxe,4.0,Unmarried,2.0,1,1,1,1.0,Manager,26576.0
4884,204884,1,28.0,Company Invited,1,31.0,Salaried,Male,4,5.0,Basic,3.0,Unmarried,3.0,1,3,1,2.0,Executive,21212.0
4885,204885,1,52.0,Self Enquiry,3,17.0,Salaried,Female,4,4.0,Standard,4.0,Married,7.0,0,1,1,3.0,Senior Manager,31820.0
4886,204886,1,19.0,Self Enquiry,3,16.0,Small Business,Male,3,4.0,Basic,3.0,Unmarried,3.0,0,5,0,2.0,Executive,20289.0


In [23]:
# Feature Engineerig

dataset['TotalNoOfPeople']=dataset['NumberOfPersonVisiting']+dataset['NumberOfChildrenVisiting']

In [24]:
dataset

,CustomerID,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,...,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome,TotalNoOfPeople
0,200000,1,41.0,Self Enquiry,3,6.0,Salaried,Female,3,3.0,...,3.0,Unmarried,1.0,1,2,1,0.0,Manager,20993.0,3.0
1,200001,0,49.0,Company Invited,1,14.0,Salaried,Male,3,4.0,...,4.0,Divorced,2.0,0,3,1,2.0,Manager,20130.0,5.0
2,200002,1,37.0,Self Enquiry,1,8.0,Free Lancer,Male,3,4.0,...,3.0,Unmarried,7.0,1,3,0,0.0,Executive,17090.0,3.0
3,200003,0,33.0,Company Invited,1,9.0,Salaried,Female,2,3.0,...,3.0,Divorced,2.0,1,5,1,1.0,Executive,17909.0,3.0
4,200004,0,36.0,Self Enquiry,1,8.0,Small Business,Male,2,3.0,...,4.0,Divorced,1.0,0,5,1,0.0,Executive,18468.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4883,204883,1,49.0,Self Enquiry,3,9.0,Small Business,Male,3,5.0,...,4.0,Unmarried,2.0,1,1,1,1.0,Manager,26576.0,4.0
4884,204884,1,28.0,Company Invited,1,31.0,Salaried,Male,4,5.0,...,3.0,Unmarried,3.0,1,3,1,2.0,Executive,21212.0,6.0
4885,204885,1,52.0,Self Enquiry,3,17.0,Salaried,Female,4,4.0,...,4.0,Married,7.0,0,1,1,3.0,Senior Manager,31820.0,7.0
4886,204886,1,19.0,Self Enquiry,3,16.0,Small Business,Male,3,4.0,...,3.0,Unmarried,3.0,0,5,0,2.0,Executive,20289.0,5.0


In [25]:
dataset.drop(['NumberOfChildrenVisiting','NumberOfPersonVisiting'],axis=1,inplace=True)

In [26]:
dataset

,CustomerID,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,Designation,MonthlyIncome,TotalNoOfPeople
0,200000,1,41.0,Self Enquiry,3,6.0,Salaried,Female,3.0,Deluxe,3.0,Unmarried,1.0,1,2,1,Manager,20993.0,3.0
1,200001,0,49.0,Company Invited,1,14.0,Salaried,Male,4.0,Deluxe,4.0,Divorced,2.0,0,3,1,Manager,20130.0,5.0
2,200002,1,37.0,Self Enquiry,1,8.0,Free Lancer,Male,4.0,Basic,3.0,Unmarried,7.0,1,3,0,Executive,17090.0,3.0
3,200003,0,33.0,Company Invited,1,9.0,Salaried,Female,3.0,Basic,3.0,Divorced,2.0,1,5,1,Executive,17909.0,3.0
4,200004,0,36.0,Self Enquiry,1,8.0,Small Business,Male,3.0,Basic,4.0,Divorced,1.0,0,5,1,Executive,18468.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4883,204883,1,49.0,Self Enquiry,3,9.0,Small Business,Male,5.0,Deluxe,4.0,Unmarried,2.0,1,1,1,Manager,26576.0,4.0
4884,204884,1,28.0,Company Invited,1,31.0,Salaried,Male,5.0,Basic,3.0,Unmarried,3.0,1,3,1,Executive,21212.0,6.0
4885,204885,1,52.0,Self Enquiry,3,17.0,Salaried,Female,4.0,Standard,4.0,Married,7.0,0,1,1,Senior Manager,31820.0,7.0
4886,204886,1,19.0,Self Enquiry,3,16.0,Small Business,Male,4.0,Basic,3.0,Unmarried,3.0,0,5,0,Executive,20289.0,5.0


In [27]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4888 entries, 0 to 4887
Data columns (total 19 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   CustomerID              4888 non-null   int64  
 1   ProdTaken               4888 non-null   int64  
 2   Age                     4888 non-null   float64
 3   TypeofContact           4888 non-null   object 
 4   CityTier                4888 non-null   int64  
 5   DurationOfPitch         4888 non-null   float64
 6   Occupation              4888 non-null   object 
 7   Gender                  4888 non-null   object 
 8   NumberOfFollowups       4888 non-null   float64
 9   ProductPitched          4888 non-null   object 
 10  PreferredPropertyStar   4888 non-null   float64
 11  MaritalStatus           4888 non-null   object 
 12  NumberOfTrips           4888 non-null   float64
 13  Passport                4888 non-null   int64  
 14  PitchSatisfactionScore  4888 non-null   

In [28]:
# Extract the number of numerical cols and categorical cols
categorical_cols=dataset.select_dtypes(include='O').columns
numerical_cols=dataset.select_dtypes(exclude='O').columns

In [29]:
categorical_cols

Index(['TypeofContact', 'Occupation', 'Gender', 'ProductPitched',
       'MaritalStatus', 'Designation'],
      dtype='object')

In [30]:
numerical_cols

Index(['CustomerID', 'ProdTaken', 'Age', 'CityTier', 'DurationOfPitch',
       'NumberOfFollowups', 'PreferredPropertyStar', 'NumberOfTrips',
       'Passport', 'PitchSatisfactionScore', 'OwnCar', 'MonthlyIncome',
       'TotalNoOfPeople'],
      dtype='object')

In [31]:
dataset

,CustomerID,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,Designation,MonthlyIncome,TotalNoOfPeople
0,200000,1,41.0,Self Enquiry,3,6.0,Salaried,Female,3.0,Deluxe,3.0,Unmarried,1.0,1,2,1,Manager,20993.0,3.0
1,200001,0,49.0,Company Invited,1,14.0,Salaried,Male,4.0,Deluxe,4.0,Divorced,2.0,0,3,1,Manager,20130.0,5.0
2,200002,1,37.0,Self Enquiry,1,8.0,Free Lancer,Male,4.0,Basic,3.0,Unmarried,7.0,1,3,0,Executive,17090.0,3.0
3,200003,0,33.0,Company Invited,1,9.0,Salaried,Female,3.0,Basic,3.0,Divorced,2.0,1,5,1,Executive,17909.0,3.0
4,200004,0,36.0,Self Enquiry,1,8.0,Small Business,Male,3.0,Basic,4.0,Divorced,1.0,0,5,1,Executive,18468.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4883,204883,1,49.0,Self Enquiry,3,9.0,Small Business,Male,5.0,Deluxe,4.0,Unmarried,2.0,1,1,1,Manager,26576.0,4.0
4884,204884,1,28.0,Company Invited,1,31.0,Salaried,Male,5.0,Basic,3.0,Unmarried,3.0,1,3,1,Executive,21212.0,6.0
4885,204885,1,52.0,Self Enquiry,3,17.0,Salaried,Female,4.0,Standard,4.0,Married,7.0,0,1,1,Senior Manager,31820.0,7.0
4886,204886,1,19.0,Self Enquiry,3,16.0,Small Business,Male,4.0,Basic,3.0,Unmarried,3.0,0,5,0,Executive,20289.0,5.0


In [32]:
# Divide the Dataset
X=dataset.drop('ProdTaken',axis=1)
y=dataset['ProdTaken']

In [33]:
X

,CustomerID,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,Designation,MonthlyIncome,TotalNoOfPeople
0,200000,41.0,Self Enquiry,3,6.0,Salaried,Female,3.0,Deluxe,3.0,Unmarried,1.0,1,2,1,Manager,20993.0,3.0
1,200001,49.0,Company Invited,1,14.0,Salaried,Male,4.0,Deluxe,4.0,Divorced,2.0,0,3,1,Manager,20130.0,5.0
2,200002,37.0,Self Enquiry,1,8.0,Free Lancer,Male,4.0,Basic,3.0,Unmarried,7.0,1,3,0,Executive,17090.0,3.0
3,200003,33.0,Company Invited,1,9.0,Salaried,Female,3.0,Basic,3.0,Divorced,2.0,1,5,1,Executive,17909.0,3.0
4,200004,36.0,Self Enquiry,1,8.0,Small Business,Male,3.0,Basic,4.0,Divorced,1.0,0,5,1,Executive,18468.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4883,204883,49.0,Self Enquiry,3,9.0,Small Business,Male,5.0,Deluxe,4.0,Unmarried,2.0,1,1,1,Manager,26576.0,4.0
4884,204884,28.0,Company Invited,1,31.0,Salaried,Male,5.0,Basic,3.0,Unmarried,3.0,1,3,1,Executive,21212.0,6.0
4885,204885,52.0,Self Enquiry,3,17.0,Salaried,Female,4.0,Standard,4.0,Married,7.0,0,1,1,Senior Manager,31820.0,7.0
4886,204886,19.0,Self Enquiry,3,16.0,Small Business,Male,4.0,Basic,3.0,Unmarried,3.0,0,5,0,Executive,20289.0,5.0


In [34]:
y

0       1
1       0
2       1
3       0
4       0
       ..
4883    1
4884    1
4885    1
4886    1
4887    1
Name: ProdTaken, Length: 4888, dtype: int64

In [35]:
# Train test Split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=42)

In [36]:
X_train

,CustomerID,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,Designation,MonthlyIncome,TotalNoOfPeople
736,200736,48.0,Self Enquiry,1,10.0,Salaried,Male,4.0,Standard,3.0,Unmarried,1.0,0,5,1,Senior Manager,25999.0,4.0
1615,201615,30.0,Self Enquiry,1,11.0,Large Business,Female,3.0,Basic,5.0,Married,6.0,0,5,0,Executive,18204.0,3.0
336,200336,29.0,Self Enquiry,1,14.0,Salaried,Male,5.0,Basic,5.0,Divorced,2.0,1,3,1,Executive,17119.0,4.0
4526,204526,29.0,Self Enquiry,3,9.0,Small Business,Female,4.0,Deluxe,4.0,Married,3.0,1,3,1,Manager,23457.0,5.0
2665,202665,34.0,Self Enquiry,1,11.0,Small Business,Female,5.0,Basic,4.0,Divorced,8.0,0,4,0,Executive,21300.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4426,204426,28.0,Self Enquiry,1,10.0,Small Business,Male,5.0,Basic,3.0,Unmarried,2.0,0,1,1,Executive,20723.0,5.0
466,200466,41.0,Self Enquiry,3,8.0,Salaried,Female,3.0,Super Deluxe,5.0,Divorced,1.0,0,5,1,AVP,31595.0,4.0
3092,203092,38.0,Company Invited,3,28.0,Small Business,Female,4.0,Basic,3.0,Divorced,7.0,0,2,1,Executive,21651.0,5.0
3772,203772,28.0,Self Enquiry,3,30.0,Small Business,Female,5.0,Deluxe,3.0,Married,3.0,0,1,1,Manager,22218.0,5.0


In [37]:
y_test

144     0
79      0
2098    0
4738    0
2858    1
       ..
2570    1
3901    0
3364    0
3639    0
1962    0
Name: ProdTaken, Length: 1467, dtype: int64

In [38]:
# Extract the number of numerical cols and categorical cols
categorical_cols=X_train.select_dtypes(include='O').columns
numerical_cols=X_train.select_dtypes(exclude='O').columns

In [39]:
categorical_cols

Index(['TypeofContact', 'Occupation', 'Gender', 'ProductPitched',
       'MaritalStatus', 'Designation'],
      dtype='object')

In [40]:

# transformer = ColumnTransformer(
#     transformers=[
#         ('oneHotEncoder', OneHotEncoder(drop='first'), categorical_cols)
#         ('Scaler',StandardScaler(),numerical_cols)
#     ],
#     remainder='passthrough'   # keeps non-categorical columns if any
# )

# # Fit and transform
# X_train_transformed = transformer.fit_transform(X_train)

# # Get column names
# encoded_cols = transformer.get_feature_names_out()

# # Convert to DataFrame with correct column names
# X_train = pd.DataFrame(
#     X_train_transformed,
#     columns=encoded_cols,
#     index=X_train.index
# )


In [41]:
# X_test_trasformed=transformer.transform(X_test)

In [42]:
X_train

,CustomerID,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,Designation,MonthlyIncome,TotalNoOfPeople
736,200736,48.0,Self Enquiry,1,10.0,Salaried,Male,4.0,Standard,3.0,Unmarried,1.0,0,5,1,Senior Manager,25999.0,4.0
1615,201615,30.0,Self Enquiry,1,11.0,Large Business,Female,3.0,Basic,5.0,Married,6.0,0,5,0,Executive,18204.0,3.0
336,200336,29.0,Self Enquiry,1,14.0,Salaried,Male,5.0,Basic,5.0,Divorced,2.0,1,3,1,Executive,17119.0,4.0
4526,204526,29.0,Self Enquiry,3,9.0,Small Business,Female,4.0,Deluxe,4.0,Married,3.0,1,3,1,Manager,23457.0,5.0
2665,202665,34.0,Self Enquiry,1,11.0,Small Business,Female,5.0,Basic,4.0,Divorced,8.0,0,4,0,Executive,21300.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4426,204426,28.0,Self Enquiry,1,10.0,Small Business,Male,5.0,Basic,3.0,Unmarried,2.0,0,1,1,Executive,20723.0,5.0
466,200466,41.0,Self Enquiry,3,8.0,Salaried,Female,3.0,Super Deluxe,5.0,Divorced,1.0,0,5,1,AVP,31595.0,4.0
3092,203092,38.0,Company Invited,3,28.0,Small Business,Female,4.0,Basic,3.0,Divorced,7.0,0,2,1,Executive,21651.0,5.0
3772,203772,28.0,Self Enquiry,3,30.0,Small Business,Female,5.0,Deluxe,3.0,Married,3.0,0,1,1,Manager,22218.0,5.0


In [43]:
# Now Both The features are Encoded Now We will Build the pipeline
transformer = ColumnTransformer(
    transformers=[
        ('oneHotEncoder', OneHotEncoder(drop='first',handle_unknown='ignore'), categorical_cols),
        ('Scaler',StandardScaler(),numerical_cols)
    ],
    remainder='passthrough'   # keeps non-categorical columns if any
)

def checkScores(models):
    cv=StratifiedKFold(shuffle=True,random_state=42)
    for m in models:
        pipe=Pipeline([
            ('preprocessing',transformer),
            ('model',m)
        ])
        print(f'----------------------{m} Model Trained-------------------')
        y_predicted=cross_val_predict(estimator=pipe,X=X_train,y=y_train,cv=cv)
        
        print(f'----->{m} Model Metricess On traing data')
        print(f'Accuracy Score: {accuracy_score(y_pred=y_predicted,y_true=y_train)}')
        print(f'Confusion Matrix:\n {confusion_matrix(y_pred=y_predicted,y_true=y_train)}')
        print(f'Classification report: \n {classification_report(y_pred=y_predicted,y_true=y_train)}')
        pipe.fit(X_train, y_train)

        y_predicted=pipe.predict(X_test)
        
        print(f'---------->{m} Model Metricess On traing data')
        print(f'Accuracy Score: {accuracy_score(y_pred=y_predicted,y_true=y_test)}')
        print(f'Confusion Matrix:\n {confusion_matrix(y_pred=y_predicted,y_true=y_test)}')
        print(f'Classification report: \n {classification_report(y_pred=y_predicted,y_true=y_test)}')

In [44]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import AdaBoostClassifier
logistic=LogisticRegression(max_iter=10000)
decision=DecisionTreeClassifier()
forest=RandomForestClassifier()
knn=KNeighborsClassifier()
ada=AdaBoostClassifier()
checkScores([logistic,decision,forest,knn,ada])

----------------------LogisticRegression(max_iter=10000) Model Trained-------------------


d:\Udemy\DataScience\ML-DL-NLP\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


----->LogisticRegression(max_iter=10000) Model Metricess On traing data
Accuracy Score: 0.8436129786612102
Confusion Matrix:
 [[2689   86]
 [ 449  197]]
Classification report: 
               precision    recall  f1-score   support

           0       0.86      0.97      0.91      2775
           1       0.70      0.30      0.42       646

    accuracy                           0.84      3421
   macro avg       0.78      0.64      0.67      3421
weighted avg       0.83      0.84      0.82      3421

---------->LogisticRegression(max_iter=10000) Model Metricess On traing data
Accuracy Score: 0.8398091342876619
Confusion Matrix:
 [[1150   43]
 [ 192   82]]
Classification report: 
               precision    recall  f1-score   support

           0       0.86      0.96      0.91      1193
           1       0.66      0.30      0.41       274

    accuracy                           0.84      1467
   macro avg       0.76      0.63      0.66      1467
weighted avg       0.82      0.84      0

d:\Udemy\DataScience\ML-DL-NLP\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


----->DecisionTreeClassifier() Model Metricess On traing data
Accuracy Score: 0.8596901490792166
Confusion Matrix:
 [[2531  244]
 [ 236  410]]
Classification report: 
               precision    recall  f1-score   support

           0       0.91      0.91      0.91      2775
           1       0.63      0.63      0.63       646

    accuracy                           0.86      3421
   macro avg       0.77      0.77      0.77      3421
weighted avg       0.86      0.86      0.86      3421

---------->DecisionTreeClassifier() Model Metricess On traing data
Accuracy Score: 0.9141104294478528
Confusion Matrix:
 [[1135   58]
 [  68  206]]
Classification report: 
               precision    recall  f1-score   support

           0       0.94      0.95      0.95      1193
           1       0.78      0.75      0.77       274

    accuracy                           0.91      1467
   macro avg       0.86      0.85      0.86      1467
weighted avg       0.91      0.91      0.91      1467

-----

...

----->RandomForestClassifier() Model Metricess On traing data
Accuracy Score: 0.8927214264834844
Confusion Matrix:
 [[2732   43]
 [ 324  322]]
Classification report: 
               precision    recall  f1-score   support

           0       0.89      0.98      0.94      2775
           1       0.88      0.50      0.64       646

    accuracy                           0.89      3421
   macro avg       0.89      0.74      0.79      3421
weighted avg       0.89      0.89      0.88      3421

---------->RandomForestClassifier() Model Metricess On traing data
Accuracy Score: 0.9134287661895024
Confusion Matrix:
 [[1186    7]
 [ 120  154]]
Classification report: 
               precision    recall  f1-score   support

           0       0.91      0.99      0.95      1193
           1       0.96      0.56      0.71       274

    accuracy                           0.91      1467
   macro avg       0.93      0.78      0.83      1467
weighted avg       0.92      0.91      0.90      1467

-----

d:\Udemy\DataScience\ML-DL-NLP\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


----->KNeighborsClassifier() Model Metricess On traing data
Accuracy Score: 0.8643671441099093
Confusion Matrix:
 [[2705   70]
 [ 394  252]]
Classification report: 
               precision    recall  f1-score   support

           0       0.87      0.97      0.92      2775
           1       0.78      0.39      0.52       646

    accuracy                           0.86      3421
   macro avg       0.83      0.68      0.72      3421
weighted avg       0.86      0.86      0.85      3421

---------->KNeighborsClassifier() Model Metricess On traing data
Accuracy Score: 0.8766189502385822
Confusion Matrix:
 [[1165   28]
 [ 153  121]]
Classification report: 
               precision    recall  f1-score   support

           0       0.88      0.98      0.93      1193
           1       0.81      0.44      0.57       274

    accuracy                           0.88      1467
   macro avg       0.85      0.71      0.75      1467
weighted avg       0.87      0.88      0.86      1467

---------

d:\Udemy\DataScience\ML-DL-NLP\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


----->AdaBoostClassifier() Model Metricess On traing data
Accuracy Score: 0.8471207249342297
Confusion Matrix:
 [[2734   41]
 [ 482  164]]
Classification report: 
               precision    recall  f1-score   support

           0       0.85      0.99      0.91      2775
           1       0.80      0.25      0.39       646

    accuracy                           0.85      3421
   macro avg       0.83      0.62      0.65      3421
weighted avg       0.84      0.85      0.81      3421

---------->AdaBoostClassifier() Model Metricess On traing data
Accuracy Score: 0.8432174505794138
Confusion Matrix:
 [[1166   27]
 [ 203   71]]
Classification report: 
               precision    recall  f1-score   support

           0       0.85      0.98      0.91      1193
           1       0.72      0.26      0.38       274

    accuracy                           0.84      1467
   macro avg       0.79      0.62      0.65      1467
weighted avg       0.83      0.84      0.81      1467



In [57]:
def tune(models):
    cv=StratifiedKFold(shuffle=True,random_state=42)
    bestModel={}
    for model, params in models.items():
        
        pipe = Pipeline([
            ('preprocessing', transformer),
            ('model', model)
        ])
        
        grid = GridSearchCV(
            estimator=pipe,
            param_grid=params,
            scoring='accuracy',
            cv=cv,
            n_jobs=-1
        )
        
        print(f'\n---------------------- {model.__class__.__name__} ----------------------')
        
        # Fit on TRAIN data only
        grid.fit(X_train, y_train)
        
        print("Best Parameters:")
        print(grid.best_params_)

        print("Best Estimators:")
        print(grid.best_estimator_)
        
        print("Best CV Accuracy:")
        print(grid.best_score_)
        
        # Test set evaluation
        y_test_pred = grid.predict(X_test)
        
        print("\n-----> Test Metrics")
        print("Accuracy:", accuracy_score(y_test, y_test_pred))
        print("Confusion Matrix:\n", confusion_matrix(y_test, y_test_pred))
        print("Classification Report:\n", classification_report(y_test, y_test_pred))
        bestModel[model.__class__.__name__]=accuracy_score(y_test, y_test_pred)
    print(bestModel)


In [58]:
logistic=LogisticRegression(max_iter=10000,
    class_weight='balanced')
decision=DecisionTreeClassifier()
forest=RandomForestClassifier()
knn=KNeighborsClassifier()
ada=AdaBoostClassifier()

logistic_params = {
    'model__C': [0.01, 0.1, 1, 10],
    'model__solver': ['liblinear', 'lbfgs']
}

decision_params = {
    'model__max_depth': [None, 5, 10, 20],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 5],
    'model__criterion': ['gini', 'entropy']
}

forest_params = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [None, 10, 20],
    'model__min_samples_split': [2, 5],
    'model__min_samples_leaf': [1, 2],
    'model__max_features': ['sqrt', 'log2']
}

knn_params = {
    'model__n_neighbors': [3, 5, 7, 9],
    'model__weights': ['uniform', 'distance'],
    'model__metric': ['euclidean', 'manhattan']
}
adaParams={
    'model__n_estimators':[50,100,500,1000],
}

models = {
    logistic: logistic_params,
    decision: decision_params,
    forest: forest_params,
    knn: knn_params,
    ada:adaParams
}


In [59]:
tune(models=models)


---------------------- LogisticRegression ----------------------
Best Parameters:
{'model__C': 10, 'model__solver': 'liblinear'}
Best Estimators:
Pipeline(steps=[('preprocessing',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('oneHotEncoder',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore'),
                                                  Index(['TypeofContact', 'Occupation', 'Gender', 'ProductPitched',
       'MaritalStatus', 'Designation'],
      dtype='object')),
                                                 ('Scaler', StandardScaler(),
                                                  Index(['CustomerID', 'Age', 'CityTier', 'DurationOfPitch', 'NumberOfFollowups',
       'PreferredPropertyStar', 'NumberOfTrips', 'Passport',
       'PitchSatisfactionScore', 'OwnCar', 'MonthlyIncome', 'TotalNoOfPeopl

In [ ]:
# From this the AdaBoost Gives 85% accuracy with best parameters